In [27]:
import os
import pandas as pd
import sys
from pathlib import Path
import numpy as np
import natsort

project_root = Path.cwd().parents[0]
sys.path.append(str(project_root))

from algorithms.tss_file import TSSFile
from algorithms.wfm_edge import WFM_edge

In [28]:
folder = r'Z:\DynamicTestData\P42025\TOLL DY_004'
output_folder = r'C:\Users\FeiyuanYang\OneDrive - Cambridge GaN Devices Ltd\Documents\CGD documents\Projects\P42025'
labels = {
    'Vg': 'ch1',
    'Is': 'ch2',
    'Vd': 'ch3',
}
chips = os.listdir(folder)
chips = [f for f in chips if "." not in f]

In [29]:
output = [['file name', 'Chip ID', 'Rg', 'Vg_LS', 'Vd', 'FF', 'dead time (us)', 'pulse width (ns)', 'temperature (C)',
           'Frame', 'VGO_LS peak (V)', 'dV/dt_neg_average_10_90 (V/ns)', 'dV/dt_neg_peak_10_90 (V/ns)', 'dV/dt_pos_average_10_90 (V/ns)', 'dV/dt_pos_peak_10_90 (V/ns)',
           'Is_peak_at_dV/dt_neg (A)', 'Is_peak_at_dV/dt_pos (A)', 
]]

In [30]:
def get_data(path, tss_file, labels,line, output):
    tss = TSSFile(path, tss_file, source='Local')
    time = tss.waveforms[labels['Vg']].time_for_frame()
    frame = tss.waveforms[labels['Vg']].frame_count
    for i in range(frame):
        if i == 0:
            continue
        vg = tss.waveforms[labels['Vg']].values_for_frame(i)
        vd = tss.waveforms[labels['Vd']].values_for_frame(i)
        current = tss.waveforms[labels['Is']].values_for_frame(i)
        vg_peak = max(vg)
        try:
            vd_edges = WFM_edge(vd, time, rising_edge_number=1, falling_edge_number=1, peak_width=1, peak_distance=10, sigma=10)
            idx_vd_fall_99 = vd_edges.get_edge(1, "falling", 99)['thresh_idx']
            idx_vd_fall_1 = vd_edges.get_edge(1, "falling", 1)['thresh_idx']
            idx_vd_fall_90 = vd_edges.get_edge(1, "falling", 90)['thresh_idx']
            idx_vd_fall_10 = vd_edges.get_edge(1, "falling", 10)['thresh_idx']

            idx_vd_rise_99 = vd_edges.get_edge(1, "rising", 99)['thresh_idx']
            idx_vd_rise_1 = vd_edges.get_edge(1, "rising", 1)['thresh_idx']
            idx_vd_rise_90 = vd_edges.get_edge(1, "rising", 90)['thresh_idx']
            idx_vd_rise_10 = vd_edges.get_edge(1, "rising", 10)['thresh_idx']

            dvdt_neg_10_90 = (vd[idx_vd_fall_10] - vd[idx_vd_fall_90]) / (time[idx_vd_fall_10] - time[idx_vd_fall_90]) * 1e-9
            dvdt_pos_10_90 = (vd[idx_vd_rise_90] - vd[idx_vd_rise_10]) / (time[idx_vd_rise_90] - time[idx_vd_rise_10]) * 1e-9
            is_peak_at_dvdt_neg = max(current[idx_vd_fall_99:idx_vd_fall_1])
            is_peak_at_dvdt_pos = max(current[idx_vd_rise_1:idx_vd_rise_99])
            #print(dvdt_neg_10_90, dvdt_pos_10_90, is_peak_at_dvdt_neg, is_peak_at_dvdt_pos)
            time_neg = time[idx_vd_fall_90:idx_vd_fall_10]
            vd_neg = vd[idx_vd_fall_90:idx_vd_fall_10]
            dvdt_neg_peak = min(np.gradient(vd_neg, time_neg)) * 1e-9
            time_pos = time[idx_vd_rise_10:idx_vd_rise_90]
            vd_pos = vd[idx_vd_rise_10:idx_vd_rise_90]
            dvdt_pos_peak = max(np.gradient(vd_pos, time_pos)) * 1e-9

            data_line = line + [i, vg_peak, dvdt_neg_10_90, dvdt_neg_peak, dvdt_pos_10_90, dvdt_pos_peak, is_peak_at_dvdt_neg, is_peak_at_dvdt_pos]
        except Exception as e:
            print(f"Error processing {tss_file} frame {i}: {e}")
            data_line = line + [i, vg_peak, '', '', '', '', '', '']

        output.append(data_line)

    return output

In [31]:
for chip in chips:
    chip_folder = os.path.join(folder, chip)
    conditions = os.listdir(chip_folder)
    conditions = natsort.natsorted(conditions)
    for condition in conditions:
        #print(chip, condition)
        pulse_width = int(condition.split('PW')[1].split('n')[0])
        dead_time = int(condition.split('time ')[1][:-1])
        if condition[-1] == 'n':
            dead_time = dead_time * 1e-3
        condition_folder = os.path.join(chip_folder, condition)
        tss_files = os.listdir(condition_folder)
        for tss_file in tss_files:
            if tss_file.endswith('.tss'):
                rg = int(tss_file.split('Rg')[1].split('-')[0])
                vd = int(tss_file.split('Vd')[1].split('-')[0])
                vg = float(tss_file.split('VG(LS)')[1].split('-')[0])
                ff = int(tss_file.split('FF')[0].split('-')[-1])
                temperature = 25 if "-HT" not in tss_file else 125
                line = [tss_file, chip, rg, vg, vd, ff, dead_time, pulse_width, temperature]
                output = get_data(condition_folder, tss_file, labels, line, output)
                print(f"Processed {tss_file} in {condition} for chip {chip}")

Processed Rg1-VG(LS)10.8-Vd400-PW75ns-DED2us-100FF-HT.tss in PW75n dead time 2u for chip AAAAF675
Processed Rg1-VG(LS)10.8-Vd400-PW75ns-DED2us-100FF.tss in PW75n dead time 2u for chip AAAAF675
Processed Rg1-VG(LS)10.8-Vd400-PW75ns-DED20us-100FF-HT.tss in PW75n dead time 20u for chip AAAAF675
Processed Rg1-VG(LS)10.8-Vd400-PW75ns-DED20us-100FF.tss in PW75n dead time 20u for chip AAAAF675
Error processing Rg1-VG(LS)10.8-Vd400-PW75ns-DED500ns-100FF-HT.tss frame 1: attempt to get argmin of an empty sequence
Error processing Rg1-VG(LS)10.8-Vd400-PW75ns-DED500ns-100FF-HT.tss frame 2: attempt to get argmin of an empty sequence
Error processing Rg1-VG(LS)10.8-Vd400-PW75ns-DED500ns-100FF-HT.tss frame 3: attempt to get argmin of an empty sequence
Error processing Rg1-VG(LS)10.8-Vd400-PW75ns-DED500ns-100FF-HT.tss frame 4: attempt to get argmin of an empty sequence
Error processing Rg1-VG(LS)10.8-Vd400-PW75ns-DED500ns-100FF-HT.tss frame 5: attempt to get argmin of an empty sequence
Error processin

C:\Users\FeiyuanYang\AppData\Local\Temp\ipykernel_15044\3261294537.py:24: RuntimeWarning: invalid value encountered in scalar divide
  dvdt_neg_10_90 = (vd[idx_vd_fall_10] - vd[idx_vd_fall_90]) / (time[idx_vd_fall_10] - time[idx_vd_fall_90]) * 1e-9


Error processing Rg1-VG(LS)10.8-Vd400-PW75ns-DED500ns-100FF.tss frame 1: max() iterable argument is empty
Error processing Rg1-VG(LS)10.8-Vd400-PW75ns-DED500ns-100FF.tss frame 2: max() iterable argument is empty
Error processing Rg1-VG(LS)10.8-Vd400-PW75ns-DED500ns-100FF.tss frame 3: max() iterable argument is empty
Error processing Rg1-VG(LS)10.8-Vd400-PW75ns-DED500ns-100FF.tss frame 4: max() iterable argument is empty
Error processing Rg1-VG(LS)10.8-Vd400-PW75ns-DED500ns-100FF.tss frame 5: index 0 is out of bounds for axis 0 with size 0
Error processing Rg1-VG(LS)10.8-Vd400-PW75ns-DED500ns-100FF.tss frame 6: max() iterable argument is empty
Error processing Rg1-VG(LS)10.8-Vd400-PW75ns-DED500ns-100FF.tss frame 7: max() iterable argument is empty
Error processing Rg1-VG(LS)10.8-Vd400-PW75ns-DED500ns-100FF.tss frame 8: max() iterable argument is empty
Error processing Rg1-VG(LS)10.8-Vd400-PW75ns-DED500ns-100FF.tss frame 9: max() iterable argument is empty
Error processing Rg1-VG(LS)10.8

KeyboardInterrupt: 

In [ ]:
output_df = pd.DataFrame(output)
output_path = os.path.join(output_folder, 'P42025_TOLL_multiframe_extracted_data.csv')
output_df.to_csv(output_path, index=False, header=False)